# 1. Data Inspection

In [19]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATA_DIR = Path("data/planetlab")
DATA_PATH = DATA_DIR / "20110420"

"""
LOAD DATASET
"""

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Directory not found: {DATA_DIR.resolve()}"
    )

files = list(DATA_DIR.rglob("*"))

for file in files:
    if file.is_file():
        print(
            f"{file.name:<50} "
            f"{file.stat().st_size / 1024:.2f} KB"
        )

planetlab_traces.zip                               5365.41 KB
146-179_surfsnel_dsl_internl_net_colostate_557     0.84 KB
146-179_surfsnel_dsl_internl_net_rnp_dcc_ufjf      0.84 KB
146-179_surfsnel_dsl_internl_net_root              0.77 KB
146-179_surfsnel_dsl_internl_net_tsinghua_xyz      0.84 KB
146-179_surfsnel_dsl_internl_net_uw_oneswarm       0.84 KB
147-179_surfsnel_dsl_internl_net_tsinghua_xyz      0.57 KB
147-179_surfsnel_dsl_internl_net_uw_oneswarm       0.57 KB
75-130-96-12_static_oxfr_ma_charter_com_irisaple_wup 0.56 KB
75-130-96-12_static_oxfr_ma_charter_com_root       0.59 KB
75-130-96-12_static_oxfr_ma_charter_com_tsinghua_xyz 0.84 KB
ait05_us_es_irisaple_wup                           0.56 KB
ait05_us_es_princeton_codeen                       0.84 KB
ait05_us_es_root                                   0.58 KB
ait05_us_es_uw_oneswarm                            0.84 KB
aladdin_planetlab_extranet_uni-passau_de_ rnp_dcc_ufjf 0.84 KB
aladdin_planetlab_extranet_uni-passau_de_tsin

# 2. Small Test Problem

In [20]:
"""
PILIH 5 FILE VM
"""
# Select 5 VM files
vm_files = sorted([
    file for file in DATA_PATH.iterdir()
    if file.is_file()
])[:5]

print("Selected VMs:")
for i, file in enumerate(vm_files):
    print(f"VM{i+1}: {file.name}")

Selected VMs:
VM1: 146-179_surfsnel_dsl_internl_net_arizona_beta
VM2: 146-179_surfsnel_dsl_internl_net_arizona_nest
VM3: 146-179_surfsnel_dsl_internl_net_howard_p2psip
VM4: 146-179_surfsnel_dsl_internl_net_root
VM5: 146-179_surfsnel_dsl_internl_net_tsinghua_xyz


In [21]:
"""
CONSTRUCT DATASET
"""
# Ensure all selected traces have equal lengths
# lengths = [len(trace) for trace in vm_traces.values()]

# if len(set(lengths)) != 1:
#     raise ValueError(
#         "Trace lengths differ. Temporal alignment must be "
#         "verified before constructing the matrix."
#     )

# vm_data = pd.DataFrame(vm_traces)
# display(vm_data.head(10))

# print("\nDataset shape:", vm_data.shape)

'\nCONSTRUCT DATASET\n'

In [22]:
# # Use the first timestamp for initial manual validation
# cpu_snapshot = vm_data.iloc[0]

# print("CPU utilization at timestamp 0:")
# display(cpu_snapshot.to_frame(name="CPU (%)"))

Pada smalltest menggunakan 3 PM dengan kapasitas CPU yg sama:
- VM = 5
- PM = 3

$$
U=[18,17,15,37,6]
$$

maka apabila menggunakan Chromosome susunan

$$
C=[0,0,1,2,1]
$$
Perhitungannya:

- PM0 = 18 + 17 = 35
- PM1 = 15 + 6 = 21
- PM2 = 37

TOTAL CPU LOAD: PM0+PM1+PM3 = 93


In [23]:
# VM CPU demand (proxy values)
cpu_demand = np.array([35, 25, 20, 15, 10])  # Example CPU demand values for 5 VMs

# Chromosome: VM -> PM
chromosome = np.array([0, 0, 1, 2, 1])

N_PM = 3

# Calculate CPU load per PM
pm_load = np.zeros(N_PM)

for vm_idx, pm_idx in enumerate(chromosome):
    pm_load[pm_idx] += cpu_demand[vm_idx]

for pm_idx, load in enumerate(pm_load):
    print(f"PM{pm_idx}: {load}")

print("Total CPU load:", pm_load.sum())

PM0: 60.0
PM1: 30.0
PM2: 15.0
Total CPU load: 105.0


Kita akan membangun dan menguji tiga hal:
1. Capacity constraint — apakah penempatan VM aman?
2. Feasibility checking — apakah sebuah chromosome valid?
3. Fitness function — seberapa bagus chromosome tersebut?

### Define Experiment Configuration
- PM CAPACITRY = 100 units
- SAFE THERSHOLD = 0.8
- SAFE CAPACITY (50 x 0.8) = 80 units

Kita ingin menyediakan safety margin sebesar 20%. Jadi meskipun PM secara fisik mampu menampung 50 unit, kita hanya mengizinkan penggunaan maksimal 40 unit.

Ingat, angka CPU VM yang kita gunakan saat ini masih merupakan proxy demand untuk eksperimen sintetis, bukan MIPS aktual.

In [24]:
import numpy as np
import pandas as pd

# Reproducibility
np.random.seed(42)

# Simulation configuration
CONFIG = {
    "n_vm": 5,
    "n_pm": 3,
    "pm_capacity": 100,
    "safe_threshold": 0.80,
}

# VM CPU demands (dummy data)
cpu_demand = np.array([35, 25, 20, 15, 10])

# Effective PM capacity
safe_capacity = (
    CONFIG["pm_capacity"] * CONFIG["safe_threshold"]
)

print("Total VM demand:", cpu_demand.sum())
print("PM physical capacity:", CONFIG["pm_capacity"])
print("PM safe capacity:", safe_capacity)

vm_df = pd.DataFrame({
    "VM": [f"VM{i+1}" for i in range(CONFIG["n_vm"])],
    "CPU Demand": cpu_demand
})
display(vm_df)
print("Total demand:", vm_df["CPU Demand"].sum())

Total VM demand: 105
PM physical capacity: 100
PM safe capacity: 80.0


,VM,CPU Demand
0,VM1,35
1,VM2,25
2,VM3,20
3,VM4,15
4,VM5,10


Total demand: 105


### Capacity Contraint

In [25]:
#########################################################
def calculate_pm_loads(chromosome, cpu_demand, n_pm):
    """
    Calculate total CPU demand assigned to each PM.
    """

    pm_loads = np.zeros(n_pm)

    for vm_idx, pm_idx in enumerate(chromosome):
        pm_loads[pm_idx] += cpu_demand[vm_idx]

    return pm_loads

chromosome = np.array([0, 0, 1, 1, 2])
    # "A": [0, 0, 0, 1, 1],
    # "B": [0, 0, 1, 1, 2],
    # "C": [0, 0, 0, 0, 1]
pm_loads = calculate_pm_loads(
    chromosome,
    cpu_demand,
    N_PM
)
print("PM Loads:", pm_loads)
#########################################################
def check_feasibility(pm_loads, safe_capacity):
    """
    Check whether all PMs satisfy the CPU capacity constraint.
    """

    overloaded = pm_loads > safe_capacity

    is_feasible = not np.any(overloaded)

    return {
        "is_feasible": is_feasible,
        "overloaded_pms": np.where(overloaded)[0],
        "max_overload": np.max(
            np.maximum(0, pm_loads - safe_capacity)
        )
    }

result = check_feasibility(pm_loads, safe_capacity)
print(result)

print("SAFE_CAPACITY:", safe_capacity)
print("PM Loads:", pm_loads)
print("Difference:", pm_loads - safe_capacity)

PM Loads: [60. 35. 10.]
{'is_feasible': True, 'overloaded_pms': array([], dtype=int64), 'max_overload': np.float64(0.0)}
SAFE_CAPACITY: 80.0
PM Loads: [60. 35. 10.]
Difference: [-20. -45. -70.]


### Calculate Objective Function

1. Menggunakan Physical Machine sesedikit mungkin, tanpa melanggar batas CPU.

In [26]:
def count_active_pms(chromosome, n_pm):
    """
    Count the number of PMs hosting at least one VM.
    """

    active_pms = np.unique(chromosome)
    return len(active_pms)

active_count = count_active_pms(chromosome, N_PM)
print("Active PMs:", active_count)
print("Inactive PMs:", N_PM - active_count)

Active PMs: 3
Inactive PMs: 0


2. Fitness Function

GA harus memilih solsui yang menggunakan sedikit server tanpa melanggar kapasitas CPU. Aturannya:
- Feasible chromosome selalu memiliki fitness lebih dari 1.
- Infeasible chromosome memiliki fitness maksimal 1.
- Di antara feasible chromosome, semakin sedikit PM aktif, semakin tinggi fitness.

In [27]:
def evaluate_chromosome(chromosome, vm_demand, config):
    # 1. Hitung total beban setiap PM
    pm_loads = calculate_pm_loads(
        chromosome,
        vm_demand,
        config["n_pm"]
    )

    # 2. Hitung utilization setiap PM dalam rasio 0–1
    pm_utilization = pm_loads / config["pm_capacity"]

    # 3. Hitung normalized overload
    rho = config["safe_threshold"]

    normalized_overload = np.maximum(
        0,
        (pm_utilization - rho) / (1 - rho)
    )

    # 4. Hitung total constraint violation
    violation = np.mean(normalized_overload ** 2)

    # 5. Tentukan apakah solusi feasible
    feasible = violation < 1e-9

    # 6. Hitung jumlah PM aktif
    active_pms = np.count_nonzero(pm_loads > 0)

    # 7. Hitung fitness
    if feasible:
        server_cost = active_pms / config["n_pm"]
        fitness = 1 + (1 / (1 + server_cost))
    else:
        server_cost = None
        fitness = 1 / (1 + violation)

    return {
        "pm_loads": pm_loads,
        "pm_utilization": pm_utilization,
        "active_pms": active_pms,
        "feasible": feasible,
        "violation": violation,
        "server_cost": server_cost,
        "fitness": fitness
    }
    
    
test_chromosomes = {
    "A": [0, 0, 0, 1, 1],
    "B": [0, 0, 1, 1, 2],
    "C": [0, 0, 0, 0, 1]
}

results = []

for name, chromosome in test_chromosomes.items():
    result = evaluate_chromosome(
        chromosome,
        cpu_demand,
        CONFIG
    )

    results.append({
        "Chromosome": name,
        "PM Loads": result["pm_loads"].tolist(),
        "Active PMs": result["active_pms"],
        "Feasible": result["feasible"],
        "Violation": result["violation"],
        "Fitness": result["fitness"]
    })

results_df = pd.DataFrame(results)
display(results_df)

,Chromosome,PM Loads,Active PMs,Feasible,Violation,Fitness
0,A,"[80.0, 25.0, 0.0]",2,True,0.0000,1.600000
1,B,"[60.0, 35.0, 10.0]",3,True,0.0000,1.500000
2,C,"[95.0, 10.0, 0.0]",2,False,0.1875,0.842105


# 3. Mathematical Validation

# 4. GA Components

# 5. GA Integration

# 6. Experiment

# 7. Full Experiment